# recount2 CLAMP models with C2CP prior — Pathway Coverage vs Sample Size (multiplier=100, max.iter=1000)

**Environment:** `clamp-analyses`

Same pipeline as `01_recount2_coverage.ipynb` with fixed
`multiplier = 100` and `max.iter = 1000` for all sample sizes.
`CLAMP_K` and seeds taken from `data/archs4/subsampled_number_of_lvs.tsv`
(same as `05_recount2_coverage_multiplier_imp.ipynb`).

Output: `output/recount2_multiplier_imp_fixed/c2cp_subsample_{N}_seed_{idx}/`

## Load libraries

In [1]:
start_time <- Sys.time()
cat("recount2 CLAMP coverage multiplier-imp-fixed analysis started at:", format(start_time), "\n")

recount2 CLAMP coverage multiplier-imp-fixed analysis started at: 2026-02-25 15:19:33 


In [2]:
if (!requireNamespace("CLAMP", quietly = TRUE)) {
    REPO_PATH <- "/home/msubirana/Documents/pivlab/CLAMP"
    remotes::install_local(REPO_PATH, force = TRUE, dependencies = FALSE)
}

library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)

source(here("config.R"))


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Matrix

Loaded glmnet 4.1-10

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



## Configuration

In [3]:
output_data_dir <- file.path(config$GENERAL$OUTPUT_DIR, "recount2_multiplier_imp_fixed")
dir.create(output_data_dir, showWarnings = FALSE, recursive = TRUE)

# C2CP prior path
data_path <- here("data", "archs4")

# Input files
out_dir   <- here("data", "recount2")
rds_file  <- file.path(out_dir, "recount2_PLIER_data", "recount_data_prep_PLIER.RDS")
rpkm_file <- file.path(out_dir, "recount2_PLIER_data", "recount_rpkm.RDS")

# Fixed hyperparameters for all runs
MULTIPLIER <- 100
MAX_ITER   <- 1000

# Load CLAMP_K and seeds per (sample_size x run) from TSV
lvs_tsv <- read.table(
    here("data", "archs4", "subsampled_number_of_lvs.tsv"),
    header     = TRUE,
    sep        = "\t",
    colClasses = c(number_of_latent_variables = "integer",
                   seed                       = "integer",
                   sample_size                = "integer")
)
lvs_tsv <- lvs_tsv[order(lvs_tsv$sample_size, lvs_tsv$seed), ]
rownames(lvs_tsv) <- NULL

sample_sizes <- sort(unique(lvs_tsv$sample_size))

block_size <- config$GENERAL$CHUNK_SIZE
N_CORES    <- config$recount2$N_CORES

message("Output dir   : ", output_data_dir)
message("Multiplier   : ", MULTIPLIER, " (fixed for all sample sizes)")
message("Max iter     : ", MAX_ITER,   " (fixed for all sample sizes)")
message("Sample sizes : ", paste(sample_sizes, collapse = ", "))
message("Total runs   : ", nrow(lvs_tsv))

Output dir   : /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2_multiplier_imp_fixed

Multiplier   : 100 (fixed for all sample sizes)

Max iter     : 1000 (fixed for all sample sizes)

Sample sizes : 500, 1000, 2000, 4000, 8000, 16000, 32000

Total runs   : 21



## Preprocess recount2 data

Identical to `00_recount2.ipynb`: Ensembl→HGNC mapping via biomaRt,
FBM creation, `preprocessCLAMPFBM`, `zscoreCLAMPFBM`.

In [4]:
preproc_genes_rds <- file.path(output_data_dir, "recount2_genes.rds")
preproc_samps_rds <- file.path(output_data_dir, "recount2_samples.rds")
preproc_fbm_rds   <- file.path(output_data_dir, "FBMrecount2_cov_preproc_filtered.rds")

if (file.exists(preproc_genes_rds) &&
    file.exists(preproc_samps_rds) &&
    file.exists(preproc_fbm_rds)) {
    message("Cached preprocessed data found — loading...")
    samples           <- readRDS(preproc_samps_rds)
    recount2_genes    <- readRDS(preproc_genes_rds)
    recount2_fbm_filt <- readRDS(preproc_fbm_rds)
    n_genes           <- nrow(recount2_fbm_filt)
    n_samps_total     <- ncol(recount2_fbm_filt)
    message("Loaded preprocessed FBM: ", n_genes, " genes x ", n_samps_total, " samples")
    PREPROCESSING_DONE <- TRUE
} else {
    message("No cache found — running full preprocessing...")
    PREPROCESSING_DONE <- FALSE
    data_prep <- readRDS(rds_file)
    meta <- list()
    meta$gene_symbols <- rownames(data_prep$rpkm.cm)
    meta$samples      <- colnames(data_prep$rpkm.cm)
    rm(data_prep)
}

No cache found — running full preprocessing...



In [5]:
if (!PREPROCESSING_DONE) {
    rpkm.df <- readRDS(rpkm_file)

    mart <- biomaRt::useDataset("hsapiens_gene_ensembl",
                                biomaRt::useMart("ensembl"))

    rpkm.df$ensembl_gene_id <- unlist(lapply(strsplit(rpkm.df$ENSG, "[.]"), `[[`, 1))

    gene.df <- biomaRt::getBM(
        filters    = "ensembl_gene_id",
        attributes = c("ensembl_gene_id", "hgnc_symbol"),
        values     = rpkm.df$ensembl_gene_id,
        mart       = mart
    )
    gene.df <- gene.df %>% dplyr::filter(complete.cases(.))

    rpkm.df <- dplyr::inner_join(gene.df, rpkm.df,
                                 by = "ensembl_gene_id",
                                 relationship = "many-to-many")

    rownames(rpkm.df) <- make.names(rpkm.df$hgnc_symbol, unique = TRUE)
    rpkm.df <- rpkm.df %>% dplyr::select(-c(ensembl_gene_id:ENSG))

    data_mat <- rpkm.df[meta$gene_symbols, meta$samples]
    rm(rpkm.df)

    data_mat <- as.matrix(as.data.table(data_mat))
}

In [6]:
if (!PREPROCESSING_DONE) {
    fbm_files <- c(
        file.path(output_data_dir, "FBMrecount2_cov.bk"),
        file.path(output_data_dir, "FBMrecount2_cov_preproc.bk"),
        file.path(output_data_dir, "FBMrecount2_cov_preproc_filtered.bk")
    )
    for (f in fbm_files) {
        if (file.exists(f)) {
            unlink(f)
            message("Removed stale FBM file: ", f)
        }
    }
}

Removed stale FBM file: /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2_multiplier_imp_fixed/FBMrecount2_cov.bk

Removed stale FBM file: /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2_multiplier_imp_fixed/FBMrecount2_cov_preproc.bk

Removed stale FBM file: /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2_multiplier_imp_fixed/FBMrecount2_cov_preproc_filtered.bk



In [7]:
if (!PREPROCESSING_DONE) {
    n_genes_raw <- length(meta$gene_symbols)
    n_samps_raw <- length(meta$samples)

    fbm_file    <- file.path(output_data_dir, "FBMrecount2_cov")
    recount2FBM <- FBM(
        nrow        = n_genes_raw,
        ncol        = n_samps_raw,
        backingfile = fbm_file,
        create_bk   = TRUE
    )
}

In [8]:
if (!PREPROCESSING_DONE) {
    n_blocks <- ceiling(n_genes_raw / block_size)
    for (i in seq_len(n_blocks)) {
        start_row <- (i - 1) * block_size + 1
        end_row   <- min(i * block_size, nrow(data_mat))
        recount2FBM[start_row:end_row, ] <- as.matrix(data_mat[start_row:end_row, ])
    }
    rm(data_mat)
    gc()
}

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,5888493,314.5,10949711,584.8,10949711,584.8
Vcells,10676604,81.5,3449043112,26314.2,4311300750,32892.7


In [9]:
if (!PREPROCESSING_DONE) {
    prep_recount2 <- preprocessCLAMPFBM(
        fbm         = recount2FBM,
        mean_cutoff = config$recount2$GENES_MEAN_CUTOFF,
        var_cutoff  = config$recount2$GENES_VAR_CUTOFF
    )

    recount2_fbm_filt <- prep_recount2$fbm_filtered
    recount2_rowStats <- prep_recount2$rowStats
}

Applying log2 transformation

Filling NAs with 0



In [10]:
if (!PREPROCESSING_DONE) {
    zscoreCLAMPFBM(recount2_fbm_filt, recount2_rowStats)
}

Applying Z-score transformation



In [11]:
if (!PREPROCESSING_DONE) {
    samples        <- meta$samples
    recount2_genes <- meta$gene_symbols[prep_recount2$kept_rows]

    n_genes       <- nrow(recount2_fbm_filt)
    n_samps_total <- ncol(recount2_fbm_filt)

    message("Preprocessed FBM: ", n_genes, " genes x ", n_samps_total, " samples")

    saveRDS(recount2_genes,    file.path(output_data_dir, "recount2_genes.rds"))
    saveRDS(samples,           file.path(output_data_dir, "recount2_samples.rds"))
    saveRDS(recount2_fbm_filt, file.path(output_data_dir, "FBMrecount2_cov_preproc_filtered.rds"))
}

Preprocessed FBM: 6000 genes x 37032 samples



## Load C2CP pathway prior

In [12]:
c2_gmt <- CLAMP:::read_gmt(file.path(data_path, "c2.cp.v2026.1.Hs.symbols.gmt"))
names(c2_gmt) <- paste0("C2CP_", names(c2_gmt))
c2_pathMat <- gmtListToSparseMat(list(C2CP = c2_gmt))
C2CP_matched <- getMatchedPathwayMat(c2_pathMat, recount2_genes)
message("Loaded and matched C2CP pathway matrix")

There are 5691 genes in the intersection between data and prior

Removing 1446 pathways

Loaded and matched C2CP pathway matrix



## Run CLAMP models — loop over sample sizes and seeds

`CLAMP_K` and seeds from TSV; `multiplier = 100` and `max.iter = 1000` fixed.

In [13]:
# Parse trace output captured from CLAMPbase / CLAMPfull to extract convergence info.
# Looks for lines like: "Progress 47 / 1000 | Bdiff=0.000312"
# and terminal messages: "Converged at ...", "not decreasing", "Converged early"
parse_convergence <- function(log_lines) {
    prog_lines <- grep("Progress", log_lines, value = TRUE)

    n_iter     <- NA_integer_
    last_Bdiff <- NA_real_

    if (length(prog_lines) > 0) {
        last_line <- tail(prog_lines, 1)

        m_iter  <- regmatches(last_line, regexpr("Progress\\s+(\\d+)", last_line))
        m_bdiff <- regmatches(last_line, regexpr("Bdiff=([0-9.eE+\\-]+)", last_line))

        if (length(m_iter)  > 0) n_iter     <- as.integer(sub("Progress\\s+", "", m_iter))
        if (length(m_bdiff) > 0) last_Bdiff <- as.numeric(sub("Bdiff=", "", m_bdiff))
    }

    converged       <- any(grepl("Converged at",    log_lines, ignore.case = TRUE))
    not_decreasing  <- any(grepl("not decreasing|Converged early", log_lines, ignore.case = TRUE))

    status <- dplyr::case_when(
        converged      ~ "converged",
        not_decreasing ~ "not_decreasing",
        TRUE           ~ "hit_max_iter"
    )

    list(n_iter     = n_iter,
         last_Bdiff = last_Bdiff,
         converged  = converged,
         status     = status)
}

In [14]:
results_summary <- data.frame(
    sample_size          = integer(),
    run                  = integer(),
    seed                 = integer(),
    n_samples            = integer(),
    CLAMP_K              = integer(),
    multiplier           = numeric(),
    max_iter             = integer(),
    n_lvs_total          = integer(),
    n_lvs_auc70          = integer(),
    n_lvs_auc90          = integer(),
    base_n_iter          = integer(),
    base_last_Bdiff      = numeric(),
    base_status          = character(),
    full_n_iter          = integer(),
    full_last_Bdiff      = numeric(),
    full_status          = character(),
    stringsAsFactors = FALSE
)

message("Total samples in preprocessed FBM: ", n_samps_total)

for (n_target in sample_sizes) {

    sub_tsv <- lvs_tsv[lvs_tsv$sample_size == n_target, ]

    for (run_idx in seq_len(nrow(sub_tsv))) {

        current_seed <- sub_tsv$seed[run_idx]
        CLAMP_K      <- sub_tsv$number_of_latent_variables[run_idx]

        message("\n", strrep("=", 60))
        message("SAMPLE SIZE: ", n_target,
                " | RUN ", run_idx, "/", nrow(sub_tsv),
                " | SEED: ", current_seed,
                " | CLAMP_K: ", CLAMP_K)
        message(strrep("=", 60))

        output_dir <- file.path(
            output_data_dir,
            paste0("c2cp_subsample_", n_target, "_seed_", run_idx)
        )
        dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

        # ── Skip if model already exists — load results instead ───────
        if (file.exists(file.path(output_dir, "CLAMPfull_C2CP.rds"))) {
            message("Output already exists — loading results.")

            existing_full <- readRDS(file.path(output_dir, "CLAMPfull_C2CP.rds"))
            existing_conv <- if (file.exists(file.path(output_dir, "convergence_info.rds")))
                                 readRDS(file.path(output_dir, "convergence_info.rds"))
                             else list(
                                 base = list(n_iter=NA_integer_, last_Bdiff=NA_real_, status=NA_character_),
                                 full = list(n_iter=NA_integer_, last_Bdiff=NA_real_, status=NA_character_)
                             )

            n_lvs_total <- nrow(existing_full$summary)
            n_lvs_auc70 <- sum(existing_full$summary$AUC >= 0.70, na.rm = TRUE)
            n_lvs_auc90 <- sum(existing_full$summary$AUC >= 0.90, na.rm = TRUE)

            results_summary <- rbind(results_summary, data.frame(
                sample_size     = n_target,
                run             = run_idx,
                seed            = current_seed,
                n_samples       = ncol(existing_full$B),
                CLAMP_K         = CLAMP_K,
                multiplier      = MULTIPLIER,
                max_iter        = MAX_ITER,
                n_lvs_total     = n_lvs_total,
                n_lvs_auc70     = n_lvs_auc70,
                n_lvs_auc90     = n_lvs_auc90,
                base_n_iter     = existing_conv$base$n_iter,
                base_last_Bdiff = existing_conv$base$last_Bdiff,
                base_status     = existing_conv$base$status,
                full_n_iter     = existing_conv$full$n_iter,
                full_last_Bdiff = existing_conv$full$last_Bdiff,
                full_status     = existing_conv$full$status
            ))

            rm(existing_full, existing_conv)
            gc()
            next
        }

        # ── Sample selection ──────────────────────────────────────────
        set.seed(current_seed)
        n_draw     <- min(n_target, n_samps_total)
        sample_idx <- sort(sample(seq_len(n_samps_total), n_draw))
        n_samples  <- length(sample_idx)
        samp_names <- samples[sample_idx]

        message("Selected ", n_samples, " samples")

        # ── Subsampled FBM ────────────────────────────────────────────
        message("Creating subsampled FBM...")
        fbm_sub_file <- file.path(output_dir, "fbm_subsampled")
        if (file.exists(paste0(fbm_sub_file, ".bk"))) unlink(paste0(fbm_sub_file, ".bk"))
        Y_sub <- big_copy(
            recount2_fbm_filt,
            ind.col     = sample_idx,
            backingfile = fbm_sub_file
        )

        # ── SVD ───────────────────────────────────────────────────────
        message("Computing SVD...")
        SVD_K <- min(n_samples - 1, n_genes - 1)

        if (N_CORES > 1) {
            options(bigstatsr.check.parallel.blas = FALSE)
            blas_nproc <- getOption("default.nproc.blas")
            options(default.nproc.blas = NULL)
        }

        svd_result <- big_randomSVD(Y_sub, k = SVD_K, ncores = N_CORES)

        if (N_CORES > 1) {
            options(bigstatsr.check.parallel.blas = TRUE)
            options(default.nproc.blas = blas_nproc)
        }

        valid_idx    <- which(!is.nan(svd_result$d))
        svd_result$d <- svd_result$d[valid_idx]
        svd_result$u <- svd_result$u[, valid_idx, drop = FALSE]
        svd_result$v <- svd_result$v[, valid_idx, drop = FALSE]

        saveRDS(svd_result, file.path(output_dir, "svd.rds"))
        saveRDS(CLAMP_K,    file.path(output_dir, "CLAMP_K.rds"))
        message("CLAMP K = ", CLAMP_K, " (from TSV)")

        saveRDS(
            list(
                sample_size     = n_target,
                run             = run_idx,
                seed            = current_seed,
                CLAMP_K         = CLAMP_K,
                multiplier      = MULTIPLIER,
                max_iter        = MAX_ITER,
                n_samples       = n_samples,
                sample_idx      = sample_idx,
                sample_names    = samp_names,
                sampling_method = "random_sampling"
            ),
            file = file.path(output_dir, "subsample_info.rds")
        )

        # ── CLAMPbase ─────────────────────────────────────────────────
        message("Running CLAMPbase...")
        base_log <- capture.output(
            baseRes <- CLAMPbase(
                Y       = Y_sub,
                svdres  = svd_result,
                trace   = TRUE,
                clamp_k = CLAMP_K
            ),
            type = "message"
        )
        base_conv <- parse_convergence(base_log)
        message(sprintf("CLAMPbase — status: %s | iter: %d | last Bdiff: %.2e",
                        base_conv$status, base_conv$n_iter, base_conv$last_Bdiff))

        baseRes$Z <- data.frame(baseRes$Z)
        rownames(baseRes$Z) <- recount2_genes
        baseRes$B <- data.frame(baseRes$B)
        colnames(baseRes$B) <- samp_names

        saveRDS(baseRes, file.path(output_dir, "CLAMPbase.rds"))

        model_dir <- file.path(output_dir, "CLAMPbase")
        dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
        write.csv(baseRes$B, file.path(model_dir, "B.csv"))
        write.csv(baseRes$Z, file.path(model_dir, "Z.csv"))

        # ── CLAMPfull with C2CP prior ─────────────────────────────────
        message("Running CLAMPfull with C2CP prior ",
                "(multiplier=", MULTIPLIER, ", max.iter=", MAX_ITER, ")...")
        full_log <- capture.output(
            fullRes <- CLAMPfull(
                Y                 = Y_sub,
                svdres            = svd_result,
                priorMat          = C2CP_matched,
                clamp.base.result = baseRes,
                use_cpp           = TRUE,
                trace             = TRUE,
                clamp_k           = CLAMP_K,
                multiplier        = MULTIPLIER,
                max.iter          = MAX_ITER
            ),
            type = "message"
        )
        full_conv <- parse_convergence(full_log)
        message(sprintf("CLAMPfull  — status: %s | iter: %d | last Bdiff: %.2e",
                        full_conv$status, full_conv$n_iter, full_conv$last_Bdiff))

        fullRes$Z <- data.frame(fullRes$Z)
        rownames(fullRes$Z) <- recount2_genes
        fullRes$B <- data.frame(fullRes$B)
        colnames(fullRes$B) <- samp_names
        fullRes$summary <- fullRes$summary %>%
            dplyr::rename(LV = LV_index) %>%
            dplyr::mutate(LV = paste0("LV", LV))

        saveRDS(fullRes, file.path(output_dir, "CLAMPfull_C2CP.rds"))

        # Save convergence logs alongside model
        saveRDS(list(base = base_conv, full = full_conv),
                file.path(output_dir, "convergence_info.rds"))
        writeLines(base_log, file.path(output_dir, "CLAMPbase_trace.txt"))
        writeLines(full_log, file.path(output_dir, "CLAMPfull_trace.txt"))

        model_dir <- file.path(output_dir, "CLAMPfull_C2CP")
        dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
        write.csv(fullRes$B,       file.path(model_dir, "B.csv"))
        write.csv(fullRes$Z,       file.path(model_dir, "Z.csv"))
        write.csv(fullRes$summary, file.path(model_dir, "summary.csv"))

        # ── Record summary ────────────────────────────────────────────
        n_lvs_total <- nrow(fullRes$summary)
        n_lvs_auc70 <- sum(fullRes$summary$AUC >= 0.70, na.rm = TRUE)
        n_lvs_auc90 <- sum(fullRes$summary$AUC >= 0.90, na.rm = TRUE)

        results_summary <- rbind(results_summary, data.frame(
            sample_size     = n_target,
            run             = run_idx,
            seed            = current_seed,
            n_samples       = n_samples,
            CLAMP_K         = CLAMP_K,
            multiplier      = MULTIPLIER,
            max_iter        = MAX_ITER,
            n_lvs_total     = n_lvs_total,
            n_lvs_auc70     = n_lvs_auc70,
            n_lvs_auc90     = n_lvs_auc90,
            base_n_iter     = base_conv$n_iter,
            base_last_Bdiff = base_conv$last_Bdiff,
            base_status     = base_conv$status,
            full_n_iter     = full_conv$n_iter,
            full_last_Bdiff = full_conv$last_Bdiff,
            full_status     = full_conv$status
        ))

        # ── Clean up ─────────────────────────────────────────────────
        rm(Y_sub, svd_result, baseRes, fullRes)
        gc()
    }
}

message("\nAll runs completed!")

Total samples in preprocessed FBM: 37032





SAMPLE SIZE: 500 | RUN 1/3 | SEED: 2876 | CLAMP_K: 30


Output already exists — loading results.



SAMPLE SIZE: 500 | RUN 2/3 | SEED: 4089 | CLAMP_K: 27


Output already exists — loading results.



SAMPLE SIZE: 500 | RUN 3/3 | SEED: 7883 | CLAMP_K: 30


Output already exists — loading results.



SAMPLE SIZE: 1000 | RUN 1/3 | SEED: 2876 | CLAMP_K: 48


Output already exists — loading results.



SAMPLE SIZE: 1000 | RUN 2/3 | SEED: 4089 | CLAMP_K: 49


Output already exists — loading results.



SAMPLE SIZE: 1000 | RUN 3/3 | SEED: 9401 | CLAMP_K: 49


Output already exists — loading results.



SAMPLE SIZE: 2000 | RUN 1/3 | SEED: 2876 | CLAMP_K: 68


Output already exists — loading results.



SAMPLE SIZE: 2000 | RUN 2/3 | SEED: 4089 | CLAMP_K: 91


Output already exists — loading results.



SAMPLE SIZE: 2000 | RUN 3/3 | SEED: 8828 | CLAMP_K: 108


Output already exists — loading results.



SAMPLE SIZE: 4000 | RUN 1/3 | SEED: 4089 | CLAMP_K: 187


Output already exists — loading r

## Save and display results summary

In [15]:
saveRDS(
    results_summary,
    file.path(output_data_dir, "subsample_results_summary_multiplier_imp_fixed.rds")
)
write.csv(
    results_summary,
    file.path(output_data_dir, "subsample_results_summary_multiplier_imp_fixed.csv"),
    row.names = FALSE
)

print(results_summary)

   sample_size run seed n_samples CLAMP_K multiplier max_iter n_lvs_total
1          500   1 2876       500      30        100     1000         257
2          500   2 4089       500      27        100     1000         232
3          500   3 7883       500      30        100     1000         244
4         1000   1 2876      1000      48        100     1000         405
5         1000   2 4089      1000      49        100     1000         402
6         1000   3 9401      1000      49        100     1000         408
7         2000   1 2876      2000      68        100     1000         538
8         2000   2 4089      2000      91        100     1000         741
9         2000   3 8828      2000     108        100     1000         828
10        4000   1 4089      4000     187        100     1000        1325
11        4000   2 7883      4000     105        100     1000         823
12        4000   3 8828      4000     156        100     1000        1155
13        8000   1 2876      8000     

In [16]:
df_convergence <- results_summary %>%
    dplyr::select(
        sample_size,
        run,
        seed,
        n_samples,
        base_converged  = base_status,
        base_n_iter,
        full_converged  = full_status,
        full_n_iter
    ) %>%
    dplyr::mutate(
        base_converged = base_converged == "converged",
        full_converged = full_converged == "converged"
    )

saveRDS(
    df_convergence,
    file.path(output_data_dir, "df_convergence.rds")
)
write.csv(
    df_convergence,
    file.path(output_data_dir, "df_convergence.csv"),
    row.names = FALSE
)

print(df_convergence)

   sample_size run seed n_samples base_converged base_n_iter full_converged
1          500   1 2876       500          FALSE          NA          FALSE
2          500   2 4089       500          FALSE          NA          FALSE
3          500   3 7883       500          FALSE          NA          FALSE
4         1000   1 2876      1000          FALSE          NA          FALSE
5         1000   2 4089      1000          FALSE          NA          FALSE
6         1000   3 9401      1000          FALSE          NA          FALSE
7         2000   1 2876      2000          FALSE          NA          FALSE
8         2000   2 4089      2000          FALSE          NA          FALSE
9         2000   3 8828      2000          FALSE          NA          FALSE
10        4000   1 4089      4000          FALSE          NA          FALSE
11        4000   2 7883      4000          FALSE          NA          FALSE
12        4000   3 8828      4000          FALSE          NA          FALSE
13        80

In [17]:
end_time     <- Sys.time()
elapsed_time <- end_time - start_time
cat("Analysis completed at:", format(end_time), "\n")
cat("Total elapsed time   :", format(elapsed_time), "\n")

Analysis completed at: 2026-02-26 02:52:31 
Total elapsed time   : 11.54949 hours 
